In [6]:
from pathlib import Path

import pandas as pd

# URL directa del dataset
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/german.csv"

# Crear carpeta data/raw si no existe
Path("data/raw").mkdir(parents=True, exist_ok=True)

# Descargar y guardar en el disco duro local
df = pd.read_csv(url, header=None)
df.to_csv("data/raw/dataset.csv", index=False)
print("¡Dataset guardado exitosamente en data/raw/dataset.csv con tamaño:", df.shape)

¡Dataset guardado exitosamente en data/raw/dataset.csv con tamaño: (1000, 21)


In [1]:
import os
import sys

import pandas as pd

# Asegurar la ruta de src para importar nuestro módulo data
sys.path.append(os.path.abspath(os.path.join('..', 'src')))
from inf8239_u01.data import download_csv

# URL directa y estable del dataset de crédito (Statlog German Credit)
URL = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/german.csv"

# 4. Descarga reproducible
path = download_csv(URL)
print(f"Dataset descargado exitosamente en: {path}")

# 5. Cargar y reconocer el esquema
df = pd.read_csv(path, header=None)
print("Dimensiones del dataset:", df.shape)
print("\nPrimeras 5 filas:")
display(df.head())
assert not df.empty, "El DataFrame está vacío"

# 6. Construir la auditoría de valores nulos y duplicados
audit = pd.DataFrame({
    "tipo": df.dtypes.astype(str),
    "ausentes": df.isna().sum(),
    "porcentaje_ausente": (df.isna().mean() * 100).round(2),
    "unicos": df.nunique(dropna=False)
}).sort_values("porcentaje_ausente", ascending=False)

print("\nDuplicados totales en el dataset:", df.duplicated().sum())
print("\nTabla de auditoría:")
display(audit)

Dataset descargado exitosamente en: data\raw\dataset.csv
Dimensiones del dataset: (1000, 21)

Primeras 5 filas:


,0,1,2,3,4,5,6,7,8,9,...,11,12,13,14,15,16,17,18,19,20
0,A11,6,A34,A43,1169,A65,A75,4,A93,A101,...,A121,67,A143,A152,2,A173,1,A192,A201,1.1
1,A12,48,A32,A43,5951,A61,A73,2,A92,A101,...,A121,22,A143,A152,1,A173,1,A191,A201,2.0
2,A14,12,A34,A46,2096,A61,A74,2,A93,A101,...,A121,49,A143,A152,1,A172,2,A191,A201,1.0
3,A11,42,A32,A42,7882,A61,A74,2,A93,A103,...,A122,45,A143,A153,1,A173,2,A191,A201,1.0
4,A11,24,A33,A40,4870,A61,A73,3,A93,A101,...,A124,53,A143,A153,2,A173,2,A191,A201,2.0



Duplicados totales en el dataset: 0

Tabla de auditoría:


,tipo,ausentes,porcentaje_ausente,unicos
0,str,0,0.0,4
1,int64,0,0.0,33
2,str,0,0.0,5
3,str,0,0.0,10
4,int64,0,0.0,921
5,str,0,0.0,5
6,str,0,0.0,5
7,int64,0,0.0,4
8,str,0,0.0,4
9,str,0,0.0,3


In [2]:
import os
from pathlib import Path

# Asegurar que la carpeta existe
os.makedirs("../src/inf8239_u01", exist_ok=True)

# Crear y guardar el archivo data.py limpiamente
with open("../src/inf8239_u01/data.py", "w", encoding="utf-8") as f:
    f.write('''from pathlib import Path
import pandas as pd

def download_csv(url: str, destination="data/raw/dataset.csv") -> Path:
    """Descarga un dataset CSV de forma reproducible."""
    if not url.startswith(("https://", "http://")):
        raise ValueError("La fuente debe ser una URL HTTP(S)")
    path = Path(destination)
    path.parent.mkdir(parents=True, exist_ok=True)
    
    frame = pd.read_csv(url)
    if frame.empty:
        raise ValueError("El dataset descargado está vacío")
    
    frame.to_csv(path, index=False)
    return path
''')

print("¡Archivo data.py creado y guardado exitosamente!")

¡Archivo data.py creado y guardado exitosamente!


In [4]:
# 7. Limpiar y definir el target correctamente (filtrando valores extraños como '1.1')
TARGET = 20

# Filtrar el DataFrame para quedarnos solo con las clases válidas (1 y 2)
df_clean = df[df[TARGET].isin([1.0, 2.0, 1, 2])].copy()

X = df_clean.drop(columns=[TARGET])
y = df_clean[TARGET].astype(int)  # Convertir a entero limpio

print("Distribución limpia del Target (y):")
print(y.value_counts())

assert not y.empty, "El DataFrame está vacío tras la limpieza"
assert y.nunique() >= 2, "El target debe tener al menos 2 clases"

# 8. Preprocesamiento con Pipeline (Numéricas y Categóricas)
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

num_cols = X.select_dtypes(include="number").columns.tolist()
cat_cols = X.select_dtypes(exclude="number").columns.tolist()

num_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scale", StandardScaler())
])

cat_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer([
    ("num", num_pipe, num_cols),
    ("cat", cat_pipe, cat_cols)
])

# 9. Dividir datos, entrenar Dummy y SVM
import sklearn.svm
from sklearn.dummy import DummyClassifier
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import train_test_split

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

dummy = Pipeline([("prep", preprocess), ("model", DummyClassifier(strategy="most_frequent"))])
svm = Pipeline([("prep", preprocess), ("model", sklearn.svm.SVC(C=1, gamma="scale", probability=True, random_state=42))])

print("\n--- Resultados de Rendimiento (F1 Macro) ---")
for name, model in {"dummy": dummy, "svm": svm}.items():
    model.fit(Xtr, ytr)
    pred = model.predict(Xte)
    print(f"Modelo {name}: {f1_score(yte, pred, average='macro'):.4f}")

print("\n--- Reporte de Clasificación del SVM ---")
print(classification_report(yte, svm.predict(Xte)))

Distribución limpia del Target (y):
20
1    699
2    300
Name: count, dtype: int64

--- Resultados de Rendimiento (F1 Macro) ---
Modelo dummy: 0.4118


c:\Users\jesus\Cursos\1_Maestria Ciencia de Datos\Ciencia de dato II\Practicas\INF8239_U01\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


Modelo svm: 0.7011

--- Reporte de Clasificación del SVM ---
              precision    recall  f1-score   support

           1       0.80      0.89      0.84       140
           2       0.66      0.48      0.56        60

    accuracy                           0.77       200
   macro avg       0.73      0.69      0.70       200
weighted avg       0.76      0.77      0.76       200

